# Recurrent Neural Networks

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

import aif

## Recurrent Neural Networks with Hidden States

In [2]:
X, Wx = torch.randn(2, 3), torch.randn(3, 4)

H, Wh = torch.randn(2, 4), torch.randn(4, 4)

In [3]:
X @ Wx + H @ Wh

tensor([[ 1.1977,  0.1925, -1.2636, -0.7102],
        [-1.0817,  1.7636, -0.1209, -0.1856]])

In [4]:
XH = torch.cat((X, H), 1)
WxWh = torch.cat((Wx, Wh), 0)

XH @ WxWh

tensor([[ 1.1977,  0.1925, -1.2636, -0.7102],
        [-1.0817,  1.7636, -0.1209, -0.1856]])

## RNN from Scratch

In [50]:
class RNNScratch(aif.Module):
    def __init__(self, num_inputs, num_hiddens, lr=0.01, sigma=0.01):
        super().__init__(lr=lr)

        self.num_inputs = num_inputs
        self.num_hiddens = num_hiddens
        self.sigma = sigma

        self.W_xh = nn.Parameter(
            torch.randn(num_inputs, num_hiddens) * sigma
        )
        self.W_hh = nn.Parameter(
            torch.randn(num_hiddens, num_hiddens) * sigma
        )
        self.b_h = nn.Parameter(torch.zeros(num_hiddens))

    def forward(self, inputs, state=None):
        if not state:
            state = torch.zeros(inputs.shape[1], num_hiddens)

        outputs = []
        for X in inputs:
            state = (
                X @ self.W_xh
                + state @ self.W_hh
                + self.b_h
            )
            outputs.append(state)

        return outputs, state

In [51]:
batch_size, num_inputs, num_hiddens, num_steps = 2, 16, 32, 100

rnn = RNNScratch(num_inputs, num_hiddens)
X = torch.ones((num_steps, batch_size, num_inputs))

outputs, state = rnn(X)

In [52]:
len(outputs), num_steps

(100, 100)

In [53]:
state.shape, (batch_size, num_hiddens)

(torch.Size([2, 32]), (2, 32))

In [54]:
outputs[0].shape, (batch_size, num_hiddens)

(torch.Size([2, 32]), (2, 32))

## Character-level RNN Language Model

In [55]:
class RNNLMScratch(aif.Classifier):
    def __init__(self, rnn, vocab_size, lr=0.01):
        super().__init__(lr=lr)
        
        self.rnn = rnn
        self.vocab_size = vocab_size
        self.lr = lr

        self.init_params()

    def init_params(self):
        self.W_hq = nn.Parameter(
            torch.randn(self.rnn.num_hiddens, self.vocab_size)
            * self.rnn.sigma
        )
        self.b_q = nn.Parameter(torch.zeros(self.vocab_size))

    def training_step(self, batch):
        l = self.loss(self(*batch[:-1]), batch[-1])
        
        return l

    def validation_step(self, batch):
        l = self.loss(self(*batch[:-1]), batch[-1])
        
        return {'loss': self.loss(y_hat, y)}

    def one_hot(self, X):
        return F.one_hot(X.T, self.vocab_size).type(torch.float32)

    def output_layer(self, rnn_outputs):
        outputs = [(H @ self.W_hq + self.b_q) for H in rnn_outputs]

        return torch.stack(outputs, 1)

    def forward(self, X, state=None):
        embs = self.one_hot(X)
        rnn_outputs, _ = self.rnn(embs, state)

        return self.output_layer(rnn_outputs)

In [57]:
model = RNNLMScratch(rnn, num_inputs)

outputs = model(torch.ones((batch_size, num_steps), dtype=torch.int64))

In [58]:
outputs.shape,  (batch_size, num_steps, num_inputs)

(torch.Size([2, 100, 16]), (2, 100, 16))